# Recycling Alloy Control with AMPL in Python
## AMPL Modeling for the Course Blending Problem

This notebook solves the same recycling alloy control problem with AMPL from Python using `amplpy`.

We solve the course-control instance where scraps A and B must be blended to produce `1000` tons of alloy at minimum cost while satisfying the composition limits.

Each run reports:
- the best solution found
- the minimum cost
- the execution time


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [8]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [9]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def extract_solution(variable, scrap_order):
    values = variable.get_values().to_dict()
    solution = {}
    for scrap in scrap_order:
        if scrap in values:
            solution[scrap] = int(round(values[scrap]))
        else:
            solution[scrap] = int(round(values[(scrap,)]))
    return solution


## Problem: Course Control Alloy Blend

Statement:

A recycling center uses two aluminum scraps, `A` and `B`, to produce a special alloy.

- Scrap `A` contains `6%` aluminum, `3%` silicon, and `4%` carbon.
- Scrap `B` contains `3%` aluminum, `6%` silicon, and `3%` carbon.
- The cost per ton is `$100` for scrap `A` and `$80` for scrap `B`.

The alloy specifications require:
- aluminum content between `3%` and `6%`
- silicon content between `3%` and `5%`
- carbon content between `3%` and `7%`

Determine the optimal mix of scraps needed to produce `1000` tons of the alloy.

In this notebook we keep integer tons so the AMPL model stays aligned with the exact-methods notebook used in class.


In [11]:
SCRAPS = ["A", "B"]
COMPONENTS = ["aluminum", "silicon", "carbon"]

COST = {
    "A": 100,
    "B": 80,
}

COMPOSITION = {
    ("A", "aluminum"): 0.06,
    ("A", "silicon"): 0.03,
    ("A", "carbon"): 0.04,
    ("B", "aluminum"): 0.03,
    ("B", "silicon"): 0.06,
    ("B", "carbon"): 0.03,
}

LOWER_SPEC = {
    "aluminum": 30,
    "silicon": 30,
    "carbon": 30,
}

UPPER_SPEC = {
    "aluminum": 60,
    "silicon": 50,
    "carbon": 70,
}

TOTAL_TONS = 1000

EXPECTED_SOLUTION = {
    "A": 334,
    "B": 666,
    "cost": 86680,
}


In [12]:
@timed
def solve_recycling_alloy_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        set SCRAPS ordered;
        set COMPONENTS ordered;

        param cost {SCRAPS} >= 0;
        param composition {SCRAPS, COMPONENTS} >= 0;
        param lower_spec {COMPONENTS} >= 0;
        param upper_spec {COMPONENTS} >= 0;
        param total_tons >= 0;

        var Use {s in SCRAPS} integer >= 0;

        minimize TotalCost:
            sum {s in SCRAPS} cost[s] * Use[s];

        subject to TotalProduction:
            sum {s in SCRAPS} Use[s] = total_tons;

        subject to MinimumComponent {c in COMPONENTS}:
            sum {s in SCRAPS} composition[s, c] * Use[s] >= lower_spec[c];

        subject to MaximumComponent {c in COMPONENTS}:
            sum {s in SCRAPS} composition[s, c] * Use[s] <= upper_spec[c];
        '''
    )

    ampl.set["SCRAPS"] = SCRAPS
    ampl.set["COMPONENTS"] = COMPONENTS
    ampl.param["cost"] = COST
    ampl.param["composition"] = COMPOSITION
    ampl.param["lower_spec"] = LOWER_SPEC
    ampl.param["upper_spec"] = UPPER_SPEC
    ampl.param["total_tons"] = TOTAL_TONS

    ampl.solve(solver=solver)

    solution = extract_solution(ampl.get_variable("Use"), SCRAPS)
    solution["cost"] = int(round(ampl.get_value("TotalCost")))
    return solution


In [13]:
ampl_result, ampl_time = solve_recycling_alloy_with_ampl()

print("=== RECYCLING ALLOY CONTROL RESULTS WITH AMPL ===")
print(f"Solution -> {ampl_result}")
print(f"Time     -> {ampl_time:.8f} seconds")

assert ampl_result == EXPECTED_SOLUTION
print("The AMPL solution matches the expected optimal blend.")


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === RECYCLING ALLOY CONTROL RESULTS WITH AMPL ===
Solution -> {'A': 334, 'B': 666, 'cost': 86680}
Time     -> 1.60347421 seconds
The AMPL solution matches the expected optimal blend.


In [14]:
aluminum = 0.06 * ampl_result["A"] + 0.03 * ampl_result["B"]
silicon = 0.03 * ampl_result["A"] + 0.06 * ampl_result["B"]
carbon = 0.04 * ampl_result["A"] + 0.03 * ampl_result["B"]

spec_report = {
    "total_tons": ampl_result["A"] + ampl_result["B"],
    "aluminum_pct": 100 * aluminum / TOTAL_TONS,
    "silicon_pct": 100 * silicon / TOTAL_TONS,
    "carbon_pct": 100 * carbon / TOTAL_TONS,
}
spec_report


{'total_tons': 1000,
 'aluminum_pct': 4.002,
 'silicon_pct': 4.998,
 'carbon_pct': 3.3340000000000005}

## Note: Integer vs Continuous Solution

If we relaxed the integrality requirement and allowed fractional tons, the optimal solution would move to the boundary of the silicon upper limit.

With `A + B = 1000`, the objective is `80000 + 20A`, so minimizing cost is equivalent to minimizing `A`.
The active restriction is:
- `0.03A + 0.06B <= 50`
- replacing `B = 1000 - A` yields `A >= 1000/3`

So the LP relaxation gives `A = 1000/3`, `B = 2000/3`, and cost `260000/3`, which is slightly lower than the integer optimum.


In [15]:
continuous_a = 1000 / 3
continuous_b = 2000 / 3
continuous_cost = 260000 / 3

continuous_report = {
    "A": continuous_a,
    "B": continuous_b,
    "cost": continuous_cost,
    "gap_vs_integer": EXPECTED_SOLUTION["cost"] - continuous_cost,
}
continuous_report


{'A': 333.3333333333333,
 'B': 666.6666666666666,
 'cost': 86666.66666666667,
 'gap_vs_integer': 13.333333333328483}